## 2. Ingest_matches_data

Lee `matches.csv` desde raw, aplica schema y escribe
`football_dev.bronze.matches` particionado por `season_year`.


In [0]:
dbutils.widgets.removeAll()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType
from pyspark.sql.functions import current_timestamp


In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "football_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlssmartdata1702")


In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/matches.csv" 


In [0]:
matches_schema = StructType(fields=[
    StructField("match_id", IntegerType(), False),
    StructField("season_year", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("match_date", DateType(), True),
    StructField("league_id", IntegerType(), True),
    StructField("home_club", StringType(), True),
    StructField("away_club", StringType(), True),
    StructField("home_goals", IntegerType(), True),
    StructField("away_goals", IntegerType(), True),
    StructField("ht_home_goals", IntegerType(), True),
    StructField("ht_away_goals", IntegerType(), True)
])


In [0]:
matches_df = spark.read \
    .option("header", True) \
    .schema(matches_schema) \
    .csv(ruta)


In [0]:
matches_final_df = matches_df.withColumn("ingestion_date", current_timestamp())


In [0]:
matches_final_df.write \
    .mode("overwrite") \
    .partitionBy("season_year") \
    .saveAsTable(f"{catalogo}.{esquema}.matches")
